# Manual Multiclass Logistic Regression Classification

This notebook follows the logistic-regression implementation in `Lab3.ipynb` and the assignment document. It uses the 300-dimensional mean Word2Vec document vectors and the `Category` column from the processed dataset.

Because the dataset has six categories, this notebook uses one multiclass logistic-regression model with a **softmax** output. Softmax produces one probability distribution whose values always add up to `1.0`. No built-in classifier is used.

The categories are imbalanced, so the manual cross-entropy gradient uses square-root class balancing. This improves overall held-out accuracy while still giving minority categories more influence than ordinary unweighted training.

In [6]:
import csv
from pathlib import Path

import numpy as np

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

LABEL_PATH = PROJECT_ROOT / "data" / "processed_data" / "preprocessed_research_papers.csv"
VECTOR_PATH = PROJECT_ROOT / "data" / "processed_data" / "document_vectors.npy"
VOCABULARY_PATH = PROJECT_ROOT / "data" / "processed_data" / "word2vec_vocabulary.csv"
WORD_VECTOR_PATH = PROJECT_ROOT / "data" / "processed_data" / "word_vectors.dat"
VECTOR_SIZE = 300

with LABEL_PATH.open("r", encoding="utf-8-sig", newline="") as label_file:
    reader = csv.DictReader(label_file)
    y = np.asarray([row["Category"].strip() for row in reader])

X = np.asarray(np.load(VECTOR_PATH, mmap_mode="r"), dtype=np.float32)

if X.shape[0] != len(y):
    raise ValueError("Vector and label row counts do not match")
if not np.isfinite(X).all():
    raise ValueError("Document vectors contain NaN or infinite values")

random_generator = np.random.default_rng(42)
train_indices = []
test_indices = []
for category in np.unique(y):
    category_indices = np.flatnonzero(y == category)
    random_generator.shuffle(category_indices)
    test_count = max(1, int(round(len(category_indices) * 0.2)))
    test_indices.extend(category_indices[:test_count])
    train_indices.extend(category_indices[test_count:])
random_generator.shuffle(train_indices)
random_generator.shuffle(test_indices)

X_train, X_test = X[train_indices], X[test_indices]
y_train, y_test = y[train_indices], y[test_indices]

feature_means = X_train.mean(axis=0, dtype=np.float64).astype(np.float32)
feature_scales = X_train.std(axis=0, dtype=np.float64).astype(np.float32)
feature_scales[feature_scales == 0] = 1.0
X_train = (X_train - feature_means) / feature_scales
X_test = (X_test - feature_means) / feature_scales

print("Mean Word2Vec feature matrix:", X.shape)
print("Categories:", np.unique(y))
print("Training rows:", len(y_train), "Testing rows:", len(y_test))

Mean Word2Vec feature matrix: (46344, 300)
Categories: ['hpc' 'iot' 'networks' 'nlp' 'security' 'vision']
Training rows: 37074 Testing rows: 9270


In [7]:
def softmax(scores):
    stable_scores = scores - np.max(scores, axis=1, keepdims=True)
    exponentials = np.exp(stable_scores)
    return exponentials / np.sum(exponentials, axis=1, keepdims=True)


def multiclass_cross_entropy(one_hot_labels, probabilities, sample_weights):
    probabilities = np.clip(probabilities, 1e-12, 1.0)
    losses = -np.sum(one_hot_labels * np.log(probabilities), axis=1)
    return np.sum(sample_weights * losses) / np.sum(sample_weights)


def train_multiclass_logistic_regression(
    X, labels, classes, learning_rate=0.1, epochs=1000
):
    sample_count, feature_count = X.shape
    class_count = len(classes)
    weights = np.zeros((feature_count, class_count))
    bias = np.zeros(class_count)
    losses = []

    label_ids = np.searchsorted(classes, labels)
    one_hot_labels = np.zeros((sample_count, class_count))
    one_hot_labels[np.arange(sample_count), label_ids] = 1.0

    class_counts = np.bincount(label_ids, minlength=class_count)
    full_balance_weights = sample_count / (class_count * class_counts)
    # Square-root balancing is a compromise: minority classes matter more,
    # but the largest class is not overwhelmed by noisy minority estimates.
    class_weights = np.sqrt(full_balance_weights)
    sample_weights = class_weights[label_ids]

    for epoch in range(epochs):
        logits = np.dot(X, weights) + bias
        probabilities = softmax(logits)
        weighted_error = sample_weights[:, None] * (probabilities - one_hot_labels)
        dw = (1 / sample_count) * np.dot(X.T, weighted_error)
        db = (1 / sample_count) * np.sum(weighted_error, axis=0)
        weights -= learning_rate * dw
        bias -= learning_rate * db
        losses.append(
            multiclass_cross_entropy(one_hot_labels, probabilities, sample_weights)
        )

    return weights, bias, losses

In [8]:
classes = np.unique(y_train)

# Train one multiclass softmax model, not independent sigmoid models.
multiclass_weights, multiclass_bias, losses = train_multiclass_logistic_regression(
    X_train, y_train, classes
)

test_logits = np.dot(X_test, multiclass_weights) + multiclass_bias
test_probabilities = softmax(test_logits)
predicted_categories = classes[np.argmax(test_probabilities, axis=1)]

print("Trained multiclass softmax model")
print("Probability row sums:", test_probabilities[:3].sum(axis=1))
print("First prediction:", y_test[0], "->", predicted_categories[0])

Trained multiclass softmax model
Probability row sums: [1. 1. 1.]
First prediction: vision -> nlp


In [9]:
accuracy = np.mean(predicted_categories == y_test)
print(f"Accuracy: {accuracy:.4f}")

for category in classes:
    true_positive = np.sum((y_test == category) & (predicted_categories == category))
    false_positive = np.sum((y_test != category) & (predicted_categories == category))
    false_negative = np.sum((y_test == category) & (predicted_categories != category))

    precision = true_positive / (true_positive + false_positive) if true_positive + false_positive else 0.0
    recall = true_positive / (true_positive + false_negative) if true_positive + false_negative else 0.0
    f1 = 2 * precision * recall / (precision + recall) if precision + recall else 0.0
    print(f"{category}: precision={precision:.4f}, recall={recall:.4f}, f1={f1:.4f}")

Accuracy: 0.3503
hpc: precision=0.4261, recall=0.2596, f1=0.3226
iot: precision=0.5233, recall=0.3993, f1=0.4530
networks: precision=0.2659, recall=0.3260, f1=0.2929
nlp: precision=0.2401, recall=0.5123, f1=0.3270
security: precision=0.1429, recall=0.0054, f1=0.0104
vision: precision=0.1456, recall=0.0269, f1=0.0454


## Try your own research-paper text

The next cell converts your title or abstract into the same averaged Word2Vec document vector used by the model. It then applies multiclass softmax, so the displayed scores form one normalized distribution and sum to `1.0`.

A correct probability calculation does not guarantee a correct category for every text. The current Word2Vec features were trained for only one pass, so the evaluation accuracy is the honest measure of model quality.

In [11]:
from collections import Counter
import re
import pandas as pd

vocabulary_table = pd.read_csv(VOCABULARY_PATH)
word_to_id = dict(zip(vocabulary_table["word"], vocabulary_table["word_id"]))
vector_row_count = max(word_to_id.values()) + 1
word_vectors = np.memmap(
    WORD_VECTOR_PATH,
    mode="r",
    dtype="float32",
    shape=(vector_row_count, VECTOR_SIZE),
)


def text_to_document_vector(text):
    tokens = re.findall(r"[a-z0-9]+(?:[-'][a-z0-9]+)*", text.lower())
    known_vectors = [word_vectors[word_to_id[token]] for token in tokens if token in word_to_id]
    if not known_vectors:
        raise ValueError("No words from this text were found in the Word2Vec vocabulary")
    return np.mean(known_vectors, axis=0).astype(np.float64)


def predict_category(text, show_probabilities=True):
    document_vector = text_to_document_vector(text)
    scaled_vector = (document_vector - feature_means) / feature_scales
    logits = np.dot(scaled_vector, multiclass_weights) + multiclass_bias
    probabilities = softmax(logits.reshape(1, -1))[0]
    assert np.isclose(probabilities.sum(), 1.0)
    predicted_category = classes[np.argmax(probabilities)]

    print("Text:", text)
    print("Predicted category:", predicted_category)
    if show_probabilities:
        print("Softmax probabilities (sum =", f"{probabilities.sum():.4f}):")
        for category, probability in sorted(zip(classes, probabilities), key=lambda item: item[1], reverse=True):
            print(f"  {category}: {probability:.4f}")
    return predicted_category, probabilities

In [21]:
user_text = input("enter user input:")
predict_category(user_text)

Text: artificial intelligence energy efficiency emission reduction industrial systems insights high-pressure air audits flue gas treatment solar plant operations
Predicted category: iot
Softmax probabilities (sum = 1.0000):
  iot: 0.9936
  networks: 0.0048
  vision: 0.0012
  hpc: 0.0003
  security: 0.0001
  nlp: 0.0000


(np.str_('iot'),
 array([2.92888853e-04, 9.93612220e-01, 4.76268577e-03, 2.24868221e-06,
        1.19692192e-04, 1.21026401e-03]))

# BERT comparison model

The following cells use built-in DistilBERT embeddings and scikit-learn logistic regression. Run them after the manual model section. The first run downloads the pretrained model, then later runs use cached embeddings.

In [12]:
import csv
from pathlib import Path

import numpy as np
import torch
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report
from sklearn.model_selection import train_test_split
from transformers.models.distilbert.modeling_distilbert import DistilBertModel
from transformers.models.distilbert.tokenization_distilbert_fast import DistilBertTokenizerFast

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
DATASET_PATH = PROJECT_ROOT / "data" / "processed_data" / "preprocessed_research_papers.csv"
CACHE_PATH = PROJECT_ROOT / "data" / "processed_data" / "distilbert_demo_embeddings.npz"
MODEL_NAME = "distilbert-base-uncased"
MAX_ROWS = 600

texts = []
labels = []
with DATASET_PATH.open("r", encoding="utf-8-sig", newline="") as data_file:
    reader = csv.DictReader(data_file)
    for row in reader:
        text = (row.get("text") or "").strip()
        label = (row.get("Category") or "").strip()
        if text and label:
            texts.append(text)
            labels.append(label)

# Use a balanced subset so the quick notebook demonstration contains every class.
labels = np.asarray(labels)
random_generator = np.random.default_rng(42)
selected_indices = []
for category in np.unique(labels):
    category_indices = np.flatnonzero(labels == category)
    random_generator.shuffle(category_indices)
    selected_indices.extend(category_indices[: MAX_ROWS // len(np.unique(labels))])
random_generator.shuffle(selected_indices)
texts = [texts[index] for index in selected_indices]
labels = labels[selected_indices]

if CACHE_PATH.exists():
    cached = np.load(CACHE_PATH, allow_pickle=False)
    embeddings = cached["embeddings"]
    print("Loaded cached DistilBERT embeddings:", embeddings.shape)
else:
    tokenizer = DistilBertTokenizerFast.from_pretrained(MODEL_NAME)
    encoder = DistilBertModel.from_pretrained(MODEL_NAME)
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    encoder.to(device).eval()
    embedding_batches = []
    with torch.no_grad():
        for start in range(0, len(texts), 8):
            batch = tokenizer(
                texts[start : start + 8],
                padding=True,
                truncation=True,
                max_length=256,
                return_tensors="pt",
            ).to(device)
            output = encoder(**batch).last_hidden_state
            mask = batch["attention_mask"].unsqueeze(-1).float()
            pooled = (output * mask).sum(1) / mask.sum(1).clamp(min=1e-9)
            embedding_batches.append(pooled.cpu().numpy())
    embeddings = np.vstack(embedding_batches).astype(np.float32)
    np.savez_compressed(CACHE_PATH, embeddings=embeddings)
    print("Created DistilBERT embeddings:", embeddings.shape)

X_train, X_test, y_train, y_test = train_test_split(
    embeddings, labels, test_size=0.2, random_state=42, stratify=labels
)
bert_classifier = LogisticRegression(
    max_iter=1000,
    class_weight="balanced",
    solver="lbfgs",
)
bert_classifier.fit(X_train, y_train)
bert_predictions = bert_classifier.predict(X_test)
print("BERT embedding shape:", embeddings.shape)
print("BERT demo accuracy:", f"{accuracy_score(y_test, bert_predictions):.4f}")
print(classification_report(y_test, bert_predictions, zero_division=0))

d:\4-1\NLP_Lab\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Created DistilBERT embeddings: (600, 768)
BERT embedding shape: (600, 768)
BERT demo accuracy: 0.6333
              precision    recall  f1-score   support

         hpc       0.52      0.55      0.54        20
         iot       0.71      0.75      0.73        20
    networks       0.60      0.45      0.51        20
         nlp       0.70      0.80      0.74        20
    security       0.65      0.75      0.70        20
      vision       0.59      0.50      0.54        20

    accuracy                           0.63       120
   macro avg       0.63      0.63      0.63       120
weighted avg       0.63      0.63      0.63       120



In [13]:
def predict_with_bert(text):
    tokenizer = DistilBertTokenizerFast.from_pretrained(MODEL_NAME)
    encoder = DistilBertModel.from_pretrained(MODEL_NAME)
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    encoder.to(device).eval()

    with torch.no_grad():
        batch = tokenizer(
            [text],
            padding=True,
            truncation=True,
            max_length=256,
            return_tensors="pt",
        ).to(device)
        output = encoder(**batch).last_hidden_state
        mask = batch["attention_mask"].unsqueeze(-1).float()
        vector = (output * mask).sum(1) / mask.sum(1).clamp(min=1e-9)

    prediction = bert_classifier.predict(vector.cpu().numpy())[0]
    probabilities = bert_classifier.predict_proba(vector.cpu().numpy())[0]
    print("Text:", text)
    print("Predicted category:", prediction)
    print("Probabilities:")
    for category, probability in sorted(
        zip(bert_classifier.classes_, probabilities),
        key=lambda item: item[1],
        reverse=True,
    ):
        print(f"  {category}: {probability:.4f}")
    return prediction, probabilities

In [31]:
user_text = input("Write a research-paper title or abstract for BERT: ")
predict_with_bert(user_text)

Text: energy efficient
Predicted category: hpc
Probabilities:
  hpc: 0.6647
  iot: 0.3335
  security: 0.0009
  vision: 0.0007
  networks: 0.0002
  nlp: 0.0000


(np.str_('hpc'),
 array([6.6470087e-01, 3.3351219e-01, 2.4199617e-04, 3.5384992e-06,
        8.5634302e-04, 6.8501203e-04], dtype=float32))